# k8a — Câu hỏi tổng hợp cho fine-tune reranker (GPU T4 ×2, vLLM)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`.
Lấy ngẫu nhiên đoạn nguồn trong kho (`N`: vi/zh/en — có PubMed vì đáp án private có), teacher `MODEL` (Qwen3-32B-AWQ,
chỉ dùng lúc huấn luyện) viết mỗi đoạn một câu hỏi tiếng Việt kiểu người dùng + bản dịch en/zh. KHÔNG dùng câu thi.
**Chạy thử trước:** `LIMIT = 100` → đọc `samples.md` + tốc độ (token/s) ở log, rồi `LIMIT = 0`.
**Đầu ra:** `synth/source.parquet`, `synth/gen.jsonl`, `qexp/synth/queries_{vi,en,zh}.parquet` → *New Dataset*
**`mir2-synth`** → k7 (`MODES = ['synth']`, `FUSED = ['synth_lin07']`, `RERANK = False`) → k8b.
**vLLM trên T4:** cài ở ô (2) (`VLLM_SPEC`, mặc định bản mới nhất). Lỗi khi nạp model → thử `VLLM_SPEC =
'vllm==0.9.2'` (hoặc bản cũ hơn có Qwen3), hoặc `MODEL = 'Qwen/Qwen3-14B-AWQ'` (nhanh ~2 lần, chất lượng thấp hơn).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8a-v1 (15/10/2026)'
N = 'vi:1400,zh:900,en:700'           # số đoạn nguồn mỗi ngôn ngữ
MODEL = 'Qwen/Qwen3-32B-AWQ'
LIMIT = 100                          # >0: chỉ N đoạn đầu (chạy thử); 0 = tất cả
VLLM_SPEC = 'vllm'
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) vLLM + đoạn nguồn (CPU)
sh(f'python -m pip install -q "{VLLM_SPEC}"', 'pip_vllm')
sh('python -c "import vllm, torch; print(vllm.__version__, torch.__version__)"')
sh(f'python -m mirret.synth sample --config {K.cfg} --n {N}', 'sample')

In [ ]:
# (3) Sinh câu hỏi (2 GPU, tensor parallel) → câu hỏi cho k7
lim = f' --limit {LIMIT}' if LIMIT else ''
sh(f'python -m mirret.synth generate --config {K.cfg} --model {MODEL} --tp 2{lim}', 'generate')
sh(f'python -m mirret.synth build --config {K.cfg}', 'build')
print(open(f'{K.work}/qexp/synth/samples.md', encoding='utf-8').read())
report('k8a')